In [5]:
from pathlib import Path
import json
import pandas as pd

# Display settings (optional)
pd.set_option('display.max_colwidth', 120)

# How many rows to show when displaying DataFrames in the notebook
pd.set_option('display.max_rows', 200)  # increase if you want more
pd.set_option('display.min_rows', 50)

In [6]:
# Choose the input JSON file (JSON Whole Model export)
file_name = "ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json"

# Default: workspace-root/JSON Whole Model/<file_name> (works when notebook is in PyDataTransform/)
input_path = (Path('..') / 'JSON Whole Model' / file_name).resolve()
if not input_path.exists():
    input_path = (Path('JSON Whole Model') / file_name).resolve()

assert input_path.exists(), f"File not found: {input_path}"

# 1) Copy the JSON being read into JSON_Edit
json_edit_dir = (Path('..') / 'JSON_Edit').resolve()
if not json_edit_dir.exists():
    json_edit_dir = (Path('JSON_Edit')).resolve()
json_edit_dir.mkdir(parents=True, exist_ok=True)

copied_path = json_edit_dir / input_path.name
copied_path.write_text(input_path.read_text(encoding='utf-8'), encoding='utf-8')
print(f"Copied source JSON to: {copied_path}")

# Keep original JSON structure in memory so we can write it back without escaping '/'
with copied_path.open('r', encoding='utf-8') as f:
    records = json.load(f)

df = pd.DataFrame(records)
working_json_path = copied_path
df.shape

Copied source JSON to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json


(16047, 4)

#### Model element Name counts
This notebook loads a `JSON Whole Model/*.json` export and prints a table with **Name**, **DbId**, **GUID**, and the **total count of that Name** across the model (duplicates included).

In [7]:
def _extract_guid(props):
    # `props` is the element's `Properties` array.
    # We look for the entry with displayName == 'GUID' (case-insensitive).
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

def _strip_guid_prefix(name):
    if pd.isna(name):
        return name
    text = str(name)
    if '_' not in text:
        return text

    prefix, rest = text.split('_', 1)
    # Remove prefixes like "1JNL9441111_" / "1JNL9442575_" (or similar ID-like prefixes)
    if len(prefix) >= 8 and any(ch.isdigit() for ch in prefix):
        return rest
    return text

def _write_json_unescaped(path, payload):
    # Python json.dumps keeps slashes as '/' (does not force '\\/')
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

# 2 & 3) Clean the Name column by removing GUID-like prefixes
df['Name'] = df['Name'].apply(_strip_guid_prefix)

# Sync cleaned names back to original JSON records
for idx, item in enumerate(records):
    if idx < len(df):
        item['Name'] = None if pd.isna(df.at[idx, 'Name']) else str(df.at[idx, 'Name'])

# Save amended JSON into JSON_Edit (does not change original source file)
_write_json_unescaped(working_json_path, records)
print(f"Updated JSON written to: {working_json_path}")

# Build GUID + key columns (GUID is primary unique key for matching)
df['GUID'] = df['Properties'].apply(_extract_guid)
df['ElementKey'] = df['GUID'].fillna(df['ExternalId'])
name_counts = df['Name'].value_counts(dropna=False)
df['NameCount'] = df['Name'].map(name_counts)

guid_duplicates = df['GUID'].dropna().duplicated().sum()
print(f"GUID duplicates found: {guid_duplicates}")

table = (
    df[['ElementKey', 'GUID', 'ExternalId', 'Name', 'DbId', 'NameCount']]
    .sort_values(['Name', 'DbId'], kind='stable')
    .reset_index(drop=True)
)

# 4) Print out the updated table
rows_to_show = 50  # set to None to show all rows (can be slow/huge)
table if rows_to_show is None else table.head(rows_to_show)

Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json
GUID duplicates found: 0


,ElementKey,GUID,ExternalId,Name,DbId,NameCount
0,d2023e11-8b64-3ab3-bdce-08c596b4a5be,d2023e11-8b64-3ab3-bdce-08c596b4a5be,0/0/2,"00-Converter AC Bus arrester, T",6,18
1,a46f2e7b-beb7-3336-90c3-4b2637c80057,a46f2e7b-beb7-3336-90c3-4b2637c80057,0/0/3,"00-Converter AC Bus arrester, T",7,18
2,74f07ba5-b9c1-3f80-b8c0-9c99c2feb031,74f07ba5-b9c1-3f80-b8c0-9c99c2feb031,0/0/4,"00-Converter AC Bus arrester, T",8,18
3,671339c9-b6df-35d2-9a61-2855394c1b75,671339c9-b6df-35d2-9a61-2855394c1b75,0/0/5,"00-Converter AC Bus arrester, T",9,18
4,f7e36a8d-7bde-393b-a1ae-f0e5a5b0a6a9,f7e36a8d-7bde-393b-a1ae-f0e5a5b0a6a9,0/0/6,"00-Converter AC Bus arrester, T",10,18
5,2c25264e-4072-3f6b-ba18-a7423480e19f,2c25264e-4072-3f6b-ba18-a7423480e19f,0/0/7,"00-Converter AC Bus arrester, T",11,18
6,4f68fa2b-59c8-3897-871d-06ef1874a3e8,4f68fa2b-59c8-3897-871d-06ef1874a3e8,0/0/8,"00-Converter AC Bus arrester, T",12,18
7,8d969705-24d0-38a2-b1eb-8bf07ff30766,8d969705-24d0-38a2-b1eb-8bf07ff30766,0/0/9,"00-Converter AC Bus arrester, T",13,18
8,290101d4-5659-365d-aca8-bc27c7ceaf28,290101d4-5659-365d-aca8-bc27c7ceaf28,0/0/10,"00-Converter AC Bus arrester, T",14,18
9,709bf2e3-2ef6-3a23-81e1-675f190eef52,709bf2e3-2ef6-3a23-81e1-675f190eef52,0/0/0/1/91,"00-Converter AC Bus arrester, T",111,18


In [8]:
# Export the table
out_dir = input_path.parent
# csv_path = out_dir / f"{input_path.stem}_name_table.csv"
xlsx_path = out_dir / f"{input_path.stem}_name_table.xlsx"

# table.to_csv(csv_path, index=False, encoding='utf-8-sig')
# print("Wrote CSV:", csv_path)

# Excel export requires openpyxl (recommended)
try:
    import openpyxl  # noqa: F401
    table.to_excel(xlsx_path, index=False)
    print("Wrote Excel:", xlsx_path)
except ImportError:
    print("Excel export skipped: package 'openpyxl' is not installed.")
    print("Run: pip install openpyxl  (or use notebook package install), then re-run this cell.")

Wrote Excel: C:\Git\APS-IFC\JSON Whole Model\ASTIDC-STAN-HE-MPD-RHP1-M-M-0001_name_table.xlsx


In [ ]:
def _strip_object_prefix(value):
    if not isinstance(value, str):
        return value
    if '_' not in value:
        return value
    prefix, rest = value.split('_', 1)
    # Trim only when prefix is exactly 12 digits followed by '_' (for example: 1NJ9372971_)
    if len(prefix) == 12 and prefix.isdigit():
        return rest
    return value

property_changes = []
objects_changed = 0

for idx, row in df.iterrows():
    object_changed = False
    rec = records[idx] if idx < len(records) else None

    guid = row.get('GUID')
    external_id = row.get('ExternalId')

    cleaned_name = _strip_object_prefix(row.get('Name'))
    if cleaned_name != row.get('Name'):
        df.at[idx, 'Name'] = cleaned_name
        if isinstance(rec, dict):
            rec['Name'] = cleaned_name
        object_changed = True

    props = rec.get('Properties') if isinstance(rec, dict) else row.get('Properties')
    if isinstance(props, list):
        for prop in props:
            if not isinstance(prop, dict):
                continue
            old_value = prop.get('value')
            new_value = _strip_object_prefix(old_value)
            if new_value != old_value:
                prop['value'] = new_value
                object_changed = True
                property_changes.append({
                    'GUID': guid,
                    'ExternalId': external_id,
                    'ObjectName': cleaned_name,
                    'DbId': row.get('DbId'),
                    'Property': prop.get('displayName'),
                    'OldValue': old_value,
                    'NewValue': new_value
                })

    if object_changed:
        objects_changed += 1

# Keep '/' in values such as ExternalId (no escaped '\\/')
_write_json_unescaped(working_json_path, records)
print(f"Updated JSON written to: {working_json_path}")

if property_changes:
    changes_df = pd.DataFrame(property_changes)
    print("\nChanged object-property values (matched by GUID first):")
    display(changes_df[['GUID', 'ExternalId', 'ObjectName', 'DbId', 'Property', 'OldValue', 'NewValue']])
else:
    print("\nNo prefixed property values found.")

print(f"\nTotal objects changed: {objects_changed}")
print(f"Total property values changed: {len(property_changes)}")

Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json

No prefixed property values found.

Total objects changed: 0
Total property values changed: 0
